In [14]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from IPython.display import clear_output
from PIL import Image
from matplotlib import cm
from time import perf_counter
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from tqdm import tqdm

warnings.filterwarnings('ignore')
import torch.nn.functional as F
plt.rc('font', size=30)

In [15]:
class Yarick_net(nn.Module):
    def __init__(self, block_type, num_blocks_list, num_classes=10):
        super(Yarick_net, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.in_planes = 64
    
        self.layer1=self.make_group(block_type[0], 64,  num_blocks_list[0], stride=1)
        self.layer2=self.make_group(block_type[1], 64,  num_blocks_list[1], stride=1)
        self.layer3=self.make_group(block_type[2], 64,  num_blocks_list[2], stride=1)
        self.layer4=self.make_group(block_type[3], 64,  num_blocks_list[3], stride=1)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * block_type[3].expansion, num_classes)

    def make_group(self, block, planes, num_blocks, stride):
            list_blocks = []
            list_blocks.append(block(self.in_planes, planes, stride))
            self.in_planes = int(planes * block.expansion)

            for _ in range(1, num_blocks):
                list_blocks.append(block(self.in_planes, planes))
            return nn.Sequential(*list_blocks)
    def forward(self, x):
            out = F.relu(self.bn1(self.conv1(x)))
            out = self.layer1(out) 
            out = self.layer2(out) 
            out = self.layer3(out)
            out = self.layer4(out) 

            out = self.avgpool(out)    
            out = out.view(out.size(0), -1) 
            out = self.fc(out)        
            
            return out

class Block_1(nn.Module): 
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
            super(Block_1, self).__init__()
            self.conv1 = nn.Conv2d(in_planes, planes*2, kernel_size=3, 
                                stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(planes*2) # Нормализация
            
            self.conv2 = nn.Conv2d(planes*2, planes, kernel_size=3, 
                                stride=1, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(planes)

            self.shortcut = nn.Sequential()
            if stride != 1 or in_planes != int(self.expansion * planes):
                self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, int(self.expansion * planes),
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(int(self.expansion * planes))
            )
            

    def forward(self, x):
            out=self.conv1(x)
            out=self.bn1(out)
            out = F.relu(out)
            
            out=self.conv2(out)
            out = self.bn2(out)
            out += self.shortcut(x)
            out = F.leaky_relu(out)
            return out
class Block_2(nn.Module): #блок который сжимает размер изображения
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
            super(Block_2, self).__init__()
            self.conv1 = nn.Conv2d(in_planes, planes//2, kernel_size=3, 
                                stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(planes//2) 
            
            self.conv2 = nn.Conv2d(planes//2, planes, kernel_size=3, 
                                stride=1, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(planes)

            self.shortcut = nn.Sequential()
            if stride != 1 or in_planes != int(self.expansion * planes):
                self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, int(self.expansion * planes),
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(int(self.expansion * planes))
            )
    def forward(self, x):
            out=self.conv1(x)
            out=self.bn1(out)
            out = F.relu(out)
            
            out=self.conv2(out)
            out = self.bn2(out)
            out += self.shortcut(x)
            out = F.leaky_relu(out)
            return out
        


class Block_3(nn.Module): 
    expansion = 0.5

    def __init__(self, in_planes, planes, stride=1):
            super(Block_3, self).__init__()
            self.conv1 = nn.Conv2d(in_planes, planes*2, kernel_size=3, 
                                stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(planes*2) 
            self.conv2 = nn.Conv2d(planes*2, int(int(self.expansion * planes)), kernel_size=3, 
                                stride=1, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(int(self.expansion * planes))

            self.shortcut = nn.Sequential()
            if stride != 1 or in_planes != int(self.expansion * planes):
                self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, int(self.expansion * planes),
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(int(self.expansion * planes))
            )
            

    def forward(self, x):
            out=self.conv1(x)
            out=self.bn1(out)
            out = F.relu(out)
            
            out=self.conv2(out)
            out = self.bn2(out)
            out += self.shortcut(x)
            out = F.leaky_relu(out)
            return out
class Block_4(nn.Module): 
    expansion = 2

    def __init__(self, in_planes, planes, stride=1):
            super(Block_4, self).__init__()
            self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, 
                                stride=stride, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(planes) 
            
            self.conv2 = nn.Conv2d(planes, int(self.expansion * planes), kernel_size=3, 
                                stride=1, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(int(self.expansion * planes))

            self.shortcut = nn.Sequential()
            if stride != 1 or in_planes != int(self.expansion * planes):
                self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, int(self.expansion * planes),
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(int(self.expansion * planes))
            )


    def forward(self, x):
            out=self.conv1(x)
            out=self.bn1(out)
            out = F.relu(out)
            
            out=self.conv2(out)
            out = self.bn2(out)
            out += self.shortcut(x)
            out = F.leaky_relu(out)
            return out


In [16]:
# Создаем модель
blocks = [Block_1, Block_2, Block_3, Block_4]
model = Yarick_net(blocks, [2, 2, 2, 2], num_classes=10)

# Тестовый прогон
test_data = torch.randn(1, 3, 32, 32) # 1 картинка RGB 32x32

output = model(test_data)
print("УРА! Сеть работает. Размер выхода:", output.shape)


УРА! Сеть работает. Размер выхода: torch.Size([1, 10])
